# Augmentation check -- both phases, 5-fold cross-validation

K-fold sibling of `both_phase_augmentation.ipynb`. That notebook's paired
comparison (baseline vs augmented) is pinned to n=8 validation patients, which
caps the per-chamber Wilcoxon tests below Holm significance no matter how
consistent the effect is (exact floor at n=8 is 2/2\*\*8 = 0.0078; the x12/x15
Holm penalty per class puts every per-chamber test out of reach). The
macro-dice endpoint already clears significance there (p=0.0078) -- this
notebook exists to see whether the EAT/RA per-chamber trends survive once the
paired sample grows from 8 patients to the full ~42-patient train+val pool.

Mechanics, not a new comparison: same phase (`'both'`), same loss (`dice_ce`),
same frozen `SPLIT_SEED=0`, same two arms (`augment=False`/`True`). The only
thing that changes is how the non-test pool is partitioned -- 5 disjoint
folds instead of one fixed 34/8 split -- via `train.split_patients_kfold()`
(see `train.py`). The held-out test set is IDENTICAL to every existing
`both_*` run at this split seed; k-fold never touches it, and the split
logic guarantees this (same patient list, same seed, same shuffle, same
test carve-out -- only the remaining pool is partitioned differently).

**Training does not happen in this kernel.** Folds are independent models
with nothing to synchronize between them, so they parallelize across GPUs as
separate OS processes -- `train.py`'s GPU selection is via
`CUDA_VISIBLE_DEVICES`, fixed once a process starts, so a single notebook
kernel is pinned to one GPU for its lifetime (see the `CUDA_VISIBLE_DEVICES`
cell in `both_phase_augmentation.ipynb`). Launch `run_both_phase_kfold.sh`
from a terminal to actually train; this notebook confirms the split, then
reads results back from disk once training has finished -- same
resume-from-disk design as the seed-based notebook.

## Setup

In [ ]:
import os

# only needed for the final test-set scoring cell at the bottom -- the split
# confirmation and pooled-analysis cells above it never touch the GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
print(f"Targeting GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")


In [ ]:
import sys
sys.path.append('.')

import json
import logging
import numpy as np
import torch
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
import train
from unet import UNet
from utils import metrics
from utils.data_loading import VolumeMRIDataset

train.dir_img = Path('./data/imgs/')
train.dir_mask = Path('./data/masks/')
train.dir_checkpoint = Path('./checkpoints/')


## Experiment configuration

`LOSS`, `PHASE` and `SPLIT_SEED` are fixed to match `both_phase_augmentation.ipynb`
and `both_phase_ablation.ipynb` exactly, so all three notebooks' results sit in
the same comparison space and share the same frozen test set. `K_FOLDS=5` is
the only new knob -- see the conversation this notebook came out of for why 5
(the pool has ~42 patients; 5 folds keeps each fold's own val set from getting
too small for its own checkpoint-selection signal, while staying cheap enough
to run: 10 runs total vs. the 6 the seed-based notebook used).

In [ ]:
PHASE      = 'both'
LOSS       = 'dice_ce'
K_FOLDS    = 5

SPLIT_SEED   = 0        # FROZEN. Identical to both_phase_augmentation.ipynb and
                         # both_phase_ablation.ipynb -- do not change; this is what
                         # keeps the test set identical across every both_* notebook.
EPOCHS       = 40
BATCH_SIZE   = 8
LR           = 1e-5
IMG_SCALE    = 1.0
N_CLASSES    = train.PHASE_N_CLASSES[PHASE]     # 6 for 'both' (bg + LV/RV/LA/RA/EAT)
N_CHANNELS   = train.PHASE_N_CHANNELS[PHASE]    # 2 for 'both' (water, fat)
AMP          = True
SELECT_ON    = 'macro_dice'
LR_SCHEDULE  = 'poly'
TEST_PERCENT = 0.15

def baseline_run_name(fold):
    return f'baseline_{LOSS}_kfold{fold}'

def aug_run_name(fold):
    return f'data_aug_{LOSS}_kfold{fold}'

VARIANTS = {'baseline': baseline_run_name, 'augmented': aug_run_name}

# train_model prefixes non-water run names with '{phase}_' internally -- see
# CLAUDE.md ("water keeps unprefixed run names"). Mirrors run_dir_for() in
# both_phase_augmentation.ipynb.
def run_dir_for(run_name):
    full_name = run_name if PHASE == 'water' else f'{PHASE}_{run_name}'
    return train.dir_checkpoint / full_name

print(f'{K_FOLDS}-fold, {EPOCHS} epochs each, baseline + augmented '
      f'({2 * K_FOLDS} runs total):')
for f in range(K_FOLDS):
    print(f'  {run_dir_for(baseline_run_name(f)).name:<32} (augment=False)')
    print(f'  {run_dir_for(aug_run_name(f)).name:<32} (augment=True)')


## Dataset

Built once, same as the seed-based notebook. `VolumeMRIDataset(..., phase='both')`
shares its on-disk cache (`data/preprocessed_cache/`, `_both`-suffixed files)
with `both_phase_augmentation.ipynb` and `both_phase_ablation.ipynb` -- already
fully populated by those notebooks' runs, so this is a cache-read, not a fresh
DICOM/NIfTI parse.

In [ ]:
dataset = VolumeMRIDataset(train.dir_img, train.dir_mask, scale=IMG_SCALE, phase=PHASE)
print(f'{len(dataset.mask_file_for)} patients, {len(dataset.index)} slices')
print(f'mask values: {dataset.mask_values}')


### Confirm the folds and the test set before spending any GPU time

Two separate things get checked here: that the 5 folds are disjoint and cover
the whole non-test pool (basic correctness of `split_patients_kfold`), and
that the test set it produces -- the one scored exactly once, at the very
end -- is byte-for-byte the same 8 patients every other `both_*` notebook at
`split_seed=0` already uses. If the second assertion ever fails, something
about the dataset or the split logic changed and the frozen test set can no
longer be trusted; do not proceed past that without finding out why.

In [ ]:
EXPECTED_TEST_PATIENTS = ['CADRE_1113', 'CADRE_1116_first', 'CADRE_1523', 'CADRE_1532',
                          'CADRE_1743', 'CADRE_1744', 'CADRE_1859', 'CADRE_1867']

all_val_patients = []
test_patients = None
for f in range(K_FOLDS):
    train_idx, val_idx, test_idx = train.split_patients_kfold(
        dataset, fold=f, k_folds=K_FOLDS, test_percent=TEST_PERCENT, seed=SPLIT_SEED
    )
    fold_test = sorted({dataset.index[i][0] for i in test_idx})
    fold_val = sorted({dataset.index[i][0] for i in val_idx})
    fold_train = sorted({dataset.index[i][0] for i in train_idx})

    if test_patients is None:
        test_patients = fold_test
    assert fold_test == test_patients, f'fold {f} has a different test set!'
    assert not (set(fold_val) & set(fold_train)), f'fold {f}: train/val overlap'
    assert not (set(fold_val) & set(fold_test)), f'fold {f}: val/test overlap'

    all_val_patients.append(set(fold_val))
    print(f'fold {f}: train {len(fold_train):>2} / val {len(fold_val):>2}   val = {fold_val}')

assert test_patients == EXPECTED_TEST_PATIENTS, \
    f'test set does not match the frozen split! got {test_patients}'

pool = set(dataset.mask_file_for) - set(test_patients)
union = set().union(*all_val_patients)
assert union == pool, 'folds do not cover the whole train+val pool'
for i in range(K_FOLDS):
    for j in range(i + 1, K_FOLDS):
        assert not (all_val_patients[i] & all_val_patients[j]), f'folds {i} and {j} overlap'

print(f'\nOK: test set matches the frozen split ({len(test_patients)} patients), '
      f'{K_FOLDS} folds are disjoint and cover the {len(pool)}-patient pool')


## Training

Not run from this kernel -- see the intro cell for why. From a terminal on a
node with GPUs free:

```bash
bash run_both_phase_kfold.sh          # uses every GPU nvidia-smi reports
bash run_both_phase_kfold.sh 0 2 3    # or pin it to specific GPU indices
```

This launches all `2 * K_FOLDS` runs as separate `train.py` processes,
distributed round-robin across whatever GPU indices you give it. Each run is
independent and resumable -- rerunning the script skips any run whose
`run_config.json` already has `'finished'`, same spirit as `run_variant()`'s
`already_done()` in `both_phase_augmentation.ipynb`. Check progress with
`nvidia-smi` / `wandb`; come back to this notebook once the runs you care
about are done -- everything below reads purely from disk, so a fresh kernel
is fine.

## Results

In [ ]:
def load_fold_rows(run_name_fn):
    """Concatenate val_metrics_per_patient.csv across every FINISHED fold.

    Unlike averaging across seeds (both_phase_augmentation.ipynb's variant_rows),
    folds are DISJOINT patients -- pooling them is a plain concatenation, not an
    average. Each patient appears in exactly one fold's val set, so this produces
    one row per (patient, class) covering the whole train+val pool, same shape as
    a single run's per-patient CSV but with ~42 patients instead of ~8.
    """
    rows, folds_found = [], []
    for f in range(K_FOLDS):
        run_dir = run_dir_for(run_name_fn(f))
        csv_path = run_dir / 'val_metrics_per_patient.csv'
        cfg_path = run_dir / 'run_config.json'
        if not (csv_path.exists() and cfg_path.exists()):
            continue
        if 'finished' not in json.loads(cfg_path.read_text()):
            continue
        rows.extend(metrics.load_per_patient(csv_path))
        folds_found.append(f)
    return rows, folds_found


pooled_rows = {}
for label, run_name_fn in VARIANTS.items():
    rows, folds_found = load_fold_rows(run_name_fn)
    pooled_rows[label] = rows
    patients = sorted({r['patient_id'] for r in rows})
    status = folds_found if len(folds_found) == K_FOLDS else f'{folds_found} (incomplete)'
    print(f'{label:<12} folds finished: {status}   {len(patients)} patients pooled')


### Per-class, side by side

Same layout as `both_phase_augmentation.ipynb`, but `mean +/- SD` is now over
the pooled train+val patients (up to ~42) instead of the 8 validation
patients -- only arms with every fold finished are shown.

In [ ]:
ORDER = ['LV', 'RV', 'LA', 'RA', 'EAT', 'all_classes_macro']

summaries = {}
for label, rows in pooled_rows.items():
    _, folds_found = load_fold_rows(VARIANTS[label])
    if len(folds_found) == K_FOLDS and rows:
        summaries[label] = {r['class_name']: r for r in metrics.summarise(rows)}

have = [v for v in VARIANTS if v in summaries]
if len(have) < len(VARIANTS):
    print('Not every arm has all K_FOLDS finished yet -- only complete arms are summarised.\n')

n_pool = len(set(dataset.mask_file_for) - set(test_patients))
for metric, unit, better in [('dice', '', 'higher'),
                             ('hd95_mm', ' mm', 'lower'),
                             ('assd_mm', ' mm', 'lower')]:
    print(f'\n=== {metric}{unit}  ({better} is better)   '
          f'mean +/- SD over up to {n_pool} pooled train+val patients ===')
    print(f'{"":<20}' + ''.join(f'{v:>24}' for v in have))
    for cls in ORDER:
        cells_ = []
        for v in have:
            r = summaries[v].get(cls)
            if r is None:
                cells_.append(f'{"n/a":>24}')
                continue
            cells_.append(f'{r[f"{metric}_mean"]:>13.4f} +/- {r[f"{metric}_sd"]:<7.4f}')
        print(f'{cls:<20}' + ''.join(cells_))


## Figures

In [ ]:
ORDER_FIG = ['LV', 'RV', 'LA', 'RA', 'EAT']
METRIC = 'dice'   # 'dice' | 'precision' | 'recall' | 'hd95_mm' | 'assd_mm'

import matplotlib.pyplot as plt

variants_present = [v for v in VARIANTS if pooled_rows.get(v)]
jitter = np.random.default_rng(0)

fig, axes = plt.subplots(1, len(ORDER_FIG), figsize=(4.3 * len(ORDER_FIG), 4.8), sharey=True)
for ax, cls in zip(axes, ORDER_FIG):
    data = [[r[METRIC] for r in pooled_rows[v]
             if r['class_name'] == cls and np.isfinite(r[METRIC])]
            for v in variants_present]

    ax.boxplot(data, tick_labels=variants_present, showmeans=True, widths=0.6)
    for i, vals in enumerate(data, start=1):
        ax.scatter(jitter.normal(i, 0.045, len(vals)), vals, alpha=0.6, s=18)
    ax.set_title(cls)
    ax.set_ylabel(METRIC if ax is axes[0] else '')
fig.suptitle(f'{METRIC} by class, pooled across {K_FOLDS} folds (baseline vs augmented)')
fig.tight_layout()
plt.show()


## Paired comparison: baseline vs augmented, pooled across folds

`compare_runs` pairs on `(patient_id, class_name)`, same test as
`both_phase_augmentation.ipynb` uses -- the difference is the shared pool is
now every train+val patient (~42), not just the 8 validation patients,
because each patient's baseline and augmented scores come from the SAME
fold's pair of models (only `augment` differs between them), so the pairing
is still valid patient-for-patient.

**Caveat, carried over from the write-up this notebook came out of:**
patients held out together in one fold are scored by that fold's one trained
model each, so they share that model's training-noise -- the pooled n is not
fully independent in the textbook sense (see Bengio & Grandvalet 2004 on the
lack of an unbiased k-fold CV variance estimator). Read the p-values below as
more informative than the 8-patient version, not as exact. The macro row is
still the single pre-specified primary endpoint; per-chamber rows are Holm-
corrected among themselves and are the thing this notebook exists to give
more power to.

In [ ]:
POOLED_DIR = train.dir_checkpoint / f'{PHASE}_baseline_vs_aug_kfold_pooled'
POOLED_DIR.mkdir(parents=True, exist_ok=True)

pooled_paths = {}
for label in VARIANTS:
    path = POOLED_DIR / f'{label}_val_metrics_per_patient.csv'
    metrics.write_csv(pooled_rows[label], path)
    pooled_paths[label] = path
print('pooled CSVs written to', POOLED_DIR)

if pooled_rows.get('baseline') and pooled_rows.get('augmented'):
    b_patients = {r['patient_id'] for r in pooled_rows['baseline']}
    a_patients = {r['patient_id'] for r in pooled_rows['augmented']}
    if b_patients != a_patients:
        print(f'WARNING: baseline has {len(b_patients)} patients pooled, augmented has '
              f'{len(a_patients)} -- compare_runs will only use the '
              f'{len(b_patients & a_patients)} patients shared by both. Wait for all '
              f'folds to finish on both arms before trusting this comparison.')

    comparison = metrics.compare_runs(pooled_paths['baseline'], pooled_paths['augmented'],
                                      label_a='baseline', label_b='augmented')
    print()
    print(metrics.format_comparison(comparison, 'baseline', 'augmented'))
else:
    print('Need at least one finished fold on both arms before comparing.')


## Diagnostics: visualize predictions across the pool

This is the piece that matters most while you're still making modeling
decisions -- you don't want to touch the test set yet, and there's no single
"the" model to load the way a fixed split had. Instead: every patient in the
train+val pool has exactly one fold whose model never trained on them --
visualize each patient with *that* fold's model. Pooled across all 5 folds,
this covers the whole ~42-patient set with genuine held-out predictions (more
than the old 8-patient val set gave you), needs no retraining, and doesn't
require deciding a "winner" first. Re-run this section after any arm finishes
training, at any decision stage -- it only reads `best.pth` files from disk.

In [ ]:
from collections import defaultdict

# which fold holds each pool patient out, once -- this never changes for a
# given SPLIT_SEED/K_FOLDS, independent of which arm you're looking at
_FOLD_OF = {}
for f in range(K_FOLDS):
    _, val_idx, _ = train.split_patients_kfold(dataset, fold=f, k_folds=K_FOLDS,
                                                test_percent=TEST_PERCENT, seed=SPLIT_SEED)
    for p in {dataset.index[i][0] for i in val_idx}:
        _FOLD_OF[p] = f

_PATIENT_IDXS = defaultdict(list)
for i, (p, _) in enumerate(dataset.index):
    _PATIENT_IDXS[p].append(i)

_model_cache = {}   # run_name -> loaded model, so re-visualizing doesn't reload weights
_pred_cache = {}    # (run_name, patient_id) -> (pred_vol, gt_vol), so re-visualizing doesn't re-infer


def load_fold_model(run_name_fn, fold):
    """Load (and cache, for this kernel) one fold's best.pth for one arm."""
    run_name = run_name_fn(fold)
    if run_name not in _model_cache:
        model = UNet(n_channels=N_CHANNELS, n_classes=N_CLASSES, bilinear=False)
        model = model.to(memory_format=torch.channels_last).to(device=device)
        state_dict = torch.load(run_dir_for(run_name) / 'best.pth', map_location=device)
        state_dict.pop('mask_values', None)      # injected by train.py, not a real weight
        model.load_state_dict(state_dict)
        model.eval()
        _model_cache[run_name] = model
    return _model_cache[run_name]


def predict_patient(patient_id, run_name_fn):
    """Predict one patient's whole volume with the ONE fold model that never
    trained on them. Raises if patient_id isn't in the train+val pool (e.g. it's
    a held-out test patient -- deliberately not reachable from here)."""
    if patient_id not in _FOLD_OF:
        raise ValueError(f'{patient_id} is not a train+val pool patient '
                         '(it may be in the held-out test set)')
    fold = _FOLD_OF[patient_id]
    run_name = run_name_fn(fold)
    key = (run_name, patient_id)
    if key not in _pred_cache:
        model = load_fold_model(run_name_fn, fold)
        pred_vol, gt_vol = metrics.predict_volume(model, dataset, _PATIENT_IDXS[patient_id],
                                                  device, amp=AMP, batch_size=BATCH_SIZE)
        _pred_cache[key] = (pred_vol, gt_vol)
    pred_vol, gt_vol = _pred_cache[key]
    return pred_vol, gt_vol, fold


def slice_scores(pred_vol, gt_vol):
    """Mean foreground Dice per slice; nan for slices with no GT foreground
    (same convention as save_best_worst_slices -- an empty slice trivially
    scores 1.0 otherwise and would swamp 'best')."""
    scores = []
    for s in range(pred_vol.shape[0]):
        cs = [metrics.dice_binary(pred_vol[s] == c, gt_vol[s] == c) for c in range(1, N_CLASSES)]
        cs = [x for x in cs if not np.isnan(x)]
        scores.append(float(np.mean(cs)) if cs else float('nan'))
    return np.array(scores)


print(f'{len(_FOLD_OF)} pool patients indexed to their held-out fold; '
      f'models/predictions load lazily and are cached per kernel session')


### Global worst/best slices, pooled across the whole CV set

Not per-fold -- across all ~42 patients at once, each scored by the model that
legitimately held them out. This is the "what isn't doing well" view; it's the
CV analogue of the auto-saved `val_worst_slice.png` a single run produces, but
searching the whole pool instead of one fold's 8-9 patients gives you more
(and more varied) failure cases to look at.

In [ ]:
def pool_slice_ranking(run_name_fn):
    """Score every slice in the pool once (predictions are cached, so re-calling
    this for the same arm is cheap). Returns entries sorted worst-to-best:
    (score, patient_id, slice_idx, fold)."""
    entries = []
    for patient_id in sorted(_FOLD_OF):
        pred_vol, gt_vol, fold = predict_patient(patient_id, run_name_fn)
        for s, score in enumerate(slice_scores(pred_vol, gt_vol)):
            if np.isfinite(score):
                entries.append((score, patient_id, s, fold))
    entries.sort(key=lambda e: e[0])
    return entries


ARM_TO_INSPECT = aug_run_name   # <- swap for baseline_run_name, or any other arm's run_name_fn

ranking = pool_slice_ranking(ARM_TO_INSPECT)
print(f'worst 6 slices, pooled across the CV set ({ARM_TO_INSPECT(0).rsplit("_kfold", 1)[0]}):')
for score, pid, s, fold in ranking[:6]:
    print(f'  {score:.3f}  {pid:<20} slice {s:<4} (fold {fold})')

print(f'\nbest 6 slices:')
for score, pid, s, fold in ranking[-6:][::-1]:
    print(f'  {score:.3f}  {pid:<20} slice {s:<4} (fold {fold})')


### Drill in: ground truth vs. one or more arms, side by side

`show_slice` takes a dict of `{label: run_name_fn}` -- pass one arm to just
look at its predictions, or two (e.g. baseline + augmented) to see exactly how
a modeling decision changed a specific case. Both panels use the SAME fold's
model for a given patient (fold membership doesn't depend on the arm), so any
visible difference is attributable to the decision, not to a different
train/val split.

In [ ]:
def show_slice(patient_id, slice_idx, arms, title_extra=''):
    """Plot ground truth + one prediction panel per arm for one slice.
    `arms` is {label: run_name_fn}."""
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch

    colors = ['none', 'tab:red', 'tab:green', 'tab:blue', 'tab:orange', 'tab:purple']
    cmap = ListedColormap(colors[:N_CLASSES])
    class_names = train.PHASE_CLASS_NAMES[PHASE]

    preds, gt, fold = {}, None, None
    for label, run_name_fn in arms.items():
        pred_vol, gt_vol, f = predict_patient(patient_id, run_name_fn)
        preds[label], gt, fold = pred_vol[slice_idx], gt_vol[slice_idx], f

    ordered = sorted(_PATIENT_IDXS[patient_id], key=lambda i: dataset.index[i][1])
    img = dataset[ordered[slice_idx]]['image'].numpy()[0]

    fig, axes = plt.subplots(1, 1 + len(arms), figsize=(4.2 * (1 + len(arms)), 4.5))
    axes = np.atleast_1d(axes)
    axes[0].imshow(img, cmap='gray')
    axes[0].imshow(np.ma.masked_equal(gt, 0), cmap=cmap, vmin=0, vmax=N_CLASSES - 1, alpha=0.5)
    axes[0].set_title('ground truth')
    axes[0].axis('off')
    for ax, (label, pred) in zip(axes[1:], preds.items()):
        ax.imshow(img, cmap='gray')
        ax.imshow(np.ma.masked_equal(pred, 0), cmap=cmap, vmin=0, vmax=N_CLASSES - 1, alpha=0.5)
        dice_here = np.nanmean([metrics.dice_binary(pred == c, gt == c) for c in range(1, N_CLASSES)])
        ax.set_title(f'{label} (Dice={dice_here:.3f})')
        ax.axis('off')

    handles = [Patch(color=colors[c], label=class_names.get(c, f'class_{c}')) for c in range(1, N_CLASSES)]
    fig.legend(handles=handles, loc='lower center', ncol=len(handles), frameon=False)
    fig.suptitle(f'{patient_id} slice {slice_idx} (fold {fold}){title_extra}')
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    plt.show()


# example: take the worst few slices found above and compare baseline vs augmented on them
for score, pid, s, fold in ranking[:3]:
    show_slice(pid, s, {'baseline': baseline_run_name, 'augmented': aug_run_name},
              title_extra=f'  [{ARM_TO_INSPECT(0).rsplit("_kfold", 1)[0]} worst-slice pick, Dice={score:.3f}]')


## Final: score the winner on the test set, once

Everything above pools validation-stage patients only; the test set has not
been touched. Run this **after** baseline vs. augmented is decided from the
pooled comparison above, and run it once.

k-fold gives you `K_FOLDS` trained models per arm, not one -- there is no
single "the" model the way a fixed single split has. Ensembling the K models
is a legitimate alternative, but it changes what "test performance" measures
(an ensemble, not a single model) and is not what any other notebook in this
project does. To stay comparable with `both_phase_augmentation.ipynb` and
`both_phase_ablation.ipynb`, this cell scores test with ONE fold's weights --
the fold with the best validation macro Dice among the winning arm -- the
same way those notebooks score test with one seed's `best.pth`.

In [ ]:
WINNING_VARIANT = 'baseline'   # <- set to 'augmented' if it won on the pooled comparison
RUN_FINAL_TEST = False          # <- flip to True deliberately, once

if RUN_FINAL_TEST:
    # pick the fold with the best validation macro Dice among the winning arm
    best_fold, best_score = None, -float('inf')
    for f in range(K_FOLDS):
        cfg_path = run_dir_for(VARIANTS[WINNING_VARIANT](f)) / 'run_config.json'
        cfg = json.loads(cfg_path.read_text())
        if cfg.get('best_val_macro_dice', -float('inf')) > best_score:
            best_fold, best_score = f, cfg['best_val_macro_dice']
    print(f'Scoring test with {WINNING_VARIANT} fold {best_fold} '
          f'(val macro Dice {best_score:.4f})')

    run_name = VARIANTS[WINNING_VARIANT](best_fold)
    run_dir = run_dir_for(run_name)
    _, _, test_idx = train.split_patients_kfold(
        dataset, fold=best_fold, k_folds=K_FOLDS, test_percent=TEST_PERCENT, seed=SPLIT_SEED
    )

    model = UNet(n_channels=N_CHANNELS, n_classes=N_CLASSES, bilinear=False)
    model = model.to(memory_format=torch.channels_last).to(device=device)

    state_dict = torch.load(run_dir / 'best.pth', map_location=device)
    state_dict.pop('mask_values', None)      # injected by train.py, not a real weight
    model.load_state_dict(state_dict)
    model.eval()

    rows, summary = metrics.report(
        model, dataset, test_idx, device,
        n_classes=N_CLASSES,
        out_dir=run_dir, split_name='test',
        class_names=train.PHASE_CLASS_NAMES[PHASE],
        batch_size=BATCH_SIZE,
    )
    print(metrics.format_summary(summary))
else:
    print('RUN_FINAL_TEST is False -- flip it deliberately once the winner is decided.')
